# GDELT (BigQuery) → extraction locale (raw)

Ce notebook exécute une requête BigQuery sur la table publique GDELT et sauvegarde le résultat dans `data/raw/`.

## Auth BigQuery (dans l'ordre)
- `GOOGLE_APPLICATION_CREDENTIALS` (chemin vers un JSON Service Account)
- `credentials/credentials.json` à la racine du repo
- sinon ADC (Application Default Credentials)

In [1]:
from __future__ import annotations
import logging
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))

import scripts.data_pipeline as dp

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s - %(message)s")
sys.path.append(str(Path().resolve().parent))

PROJECT_ROOT = Path().resolve().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import importlib
# Exécute le fichier data_pipeline.py et le recharge afin de s’assurer que les dernières modifications sont prises en compte dans la session en cours.
importlib.reload(dp)

<module 'scripts.data_pipeline' from 'C:\\Users\\DELL latitude 5420\\Documents\\Hackathon_iSHEEROXDatacamp\\scripts\\data_pipeline.py'>

In [ ]:
events_query = """
SELECT *
FROM `gdelt-bq.gdeltv2.events` AS e
WHERE (
        (e.ActionGeo_CountryCode = 'BN' 
         AND LOWER(RTRIM(e.ActionGeo_FullName)) = 'benin' 
         AND e.ActionGeo_Type = 1)
     OR (e.Actor1Geo_CountryCode = 'BN' 
         AND LOWER(RTRIM(e.Actor1Geo_FullName)) = 'benin' 
         AND e.Actor1Geo_Type = 1)
     OR (e.Actor2Geo_CountryCode = 'BN' 
         AND LOWER(RTRIM(e.Actor2Geo_FullName)) = 'benin' 
         AND e.Actor2Geo_Type = 1)
      )
  AND e.SQLDATE BETWEEN 20250101 AND 20251231
""".strip()

client = dp.get_bq_client()
df = dp.extract_raw_data(client, events_query)

raw_dir = PROJECT_ROOT / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)
dp.load_raw_data(df, raw_dir / "gdelt_bn_2025.csv")
df.head()

In [ ]:
job = client.query(events_query)
print(job.state)
print(job.error_result)

In [3]:
gkg_query = """
SELECT 
    g.Date, 
    g.SourceCommonName, 
    g.Persons, 
    g.Organizations, 
    g.Locations, 
    g.Counts,
    g.TranslationInfo
FROM `gdelt-bq.gdeltv2.gkg` AS g
INNER JOIN `gdelt-bq.gdeltv2.events` AS e
  ON g.DocumentIdentifier = e.SOURCEURL
  AND DIV(g.Date, 1000000) = e.SQLDATE
WHERE e.SQLDATE BETWEEN 20250101 AND 20251231
  AND g.SourceCollectionIdentifier = 1
  AND (
        (e.ActionGeo_CountryCode = 'BN' AND LOWER(RTRIM(e.ActionGeo_FullName)) = 'benin' AND e.ActionGeo_Type = 1)
     OR (e.Actor1Geo_CountryCode = 'BN' AND LOWER(RTRIM(e.Actor1Geo_FullName)) = 'benin' AND e.Actor1Geo_Type = 1)
     OR (e.Actor2CountryCode = 'BN' AND LOWER(RTRIM(e.Actor2Geo_FullName)) = 'benin' AND e.Actor2Geo_Type = 1)
  )
""".strip()

## Recupération des données GKG
client = dp.get_bq_client()
raw_dir = PROJECT_ROOT / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

try:
  # Exécution de la requête
  job = client.query(gkg_query)
  job.result()  # force l'attente de la fin
  print("État du job :", job.state)

  # Extraction des données
  gkg_df = dp.extract_raw_data(client, gkg_query)

  # Sauvegarde du DataFrame dans un fichier CSV
  output_file = raw_dir / "gdelt_gkg_bn_2025.csv"
  dp.load_raw_data(gkg_df, output_file)

  # Vérification rapide
  print("Aperçu des données :")
  print(gkg_df.head())
except Exception as e:
  print("Erreur rencontrée :", e)
  if "job" in locals():
    print("Détails du job :", getattr(job, "error_result", None))


2026-05-03 22:00:02,861 INFO scripts.data_pipeline - Using service account credentials from E:\junior\Formations\Hackathons\Hackathon_iSHEEROXDatacamp\credentials\credentials.json (project=black-vehicle-495220-u2)
2026-05-03 22:00:06,031 INFO scripts.data_pipeline - Running query (729 chars)


État du job : DONE


2026-05-03 22:00:50,821 INFO scripts.data_pipeline - DataFrame saved to E:\junior\Formations\Hackathons\Hackathon_iSHEEROXDatacamp\data\raw\gdelt_gkg_bn_2025.csv


Aperçu des données :
             Date       SourceCommonName  \
0  20251012024500        myjoyonline.com   
1  20251012030000  thenationonlineng.net   
2  20251012030000  thenationonlineng.net   
3  20251012190000             jagran.com   
4  20251012220000            thecable.ng   

                                             Persons  \
0                                                NaN   
1                         gloria bakare;ebonny musik   
2                         gloria bakare;ebonny musik   
3  sivan a sun sharma;ashok mahato;gazipur a chau...   
4  frank azuekor;bayo onanuga;mike ozekhome;bola ...   

                                       Organizations  \
0                   ghana meteorological agency gmet   
1  starburg empowerment foundation;university of ...   
2  starburg empowerment foundation;university of ...   
3  visa the company a operations;visa;group the c...   
4  national open university;department of state s...   

                                        